# AssureX Claim Engine — Model Training Notebook

**Project:** AssureX — AI-Powered Warranty Claim Validation System  
**Model:** Random Forest Classifier (Claim Decision + Image Classifier)  
**Dataset:** 1,500 warranty claims (500 Valid / 500 Invalid / 500 Manual Review)  
**Split:** 70% Train / 15% Validation / 15% Test  

---

## Instructions
1. Upload `train.csv`, `validation.csv`, `test.csv` to this Colab session (Files panel on left)
2. Run all cells in order
3. Download `assurex_claim_decision_model.pkl` at the end

## Step 1: Install Libraries

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib openpyxl -q
print("Libraries installed successfully!")

## Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported!")
import sklearn
print(f"scikit-learn version: {sklearn.__version__}")

## Step 3: Load Dataset (Pre-Split CSV Files)

> Upload `train.csv`, `validation.csv`, `test.csv` from your `dataset/` folder before running this cell.

In [ ]:
# Load pre-split datasets
train_df = pd.read_csv('train.csv')
val_df   = pd.read_csv('validation.csv')
test_df  = pd.read_csv('test.csv')

print("Dataset loaded successfully!")
print(f"  Training rows:    {len(train_df)}")
print(f"  Validation rows:  {len(val_df)}")
print(f"  Test rows:        {len(test_df)}")
print(f"  Total:            {len(train_df)+len(val_df)+len(test_df)}")
print(f"\nColumns: {list(train_df.columns)}")

## Step 4: Explore the Dataset

In [ ]:
print("=== Training Data Sample ===")
display(train_df.head())

print("\n=== Class Distribution (Training Set) ===")
print(train_df['Claim_Decision'].value_counts())

# Plot class distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (df, title) in zip(axes, [
    (train_df, 'Training Set'),
    (val_df,   'Validation Set'),
    (test_df,  'Test Set')
]):
    counts = df['Claim_Decision'].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=['#4CAF50','#F44336','#FF9800'])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(val), ha='center', fontsize=10)

plt.suptitle('AssureX Dataset — Class Distribution per Split',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved: class_distribution.png")

## Step 5: Feature Engineering

In [ ]:
TARGET = 'Claim_Decision'

# Separate features and labels
def prepare(df):
    df = df.drop(columns=['Claim_ID'], errors='ignore')
    X = df.drop(columns=[TARGET])
    y = df[TARGET].astype(str)
    return X, y

X_train_raw, y_train = prepare(train_df)
X_val_raw,   y_val   = prepare(val_df)
X_test_raw,  y_test  = prepare(test_df)

# One-hot encode — align columns across all splits
X_all = pd.get_dummies(pd.concat([X_train_raw, X_val_raw, X_test_raw]))
n_train = len(X_train_raw)
n_val   = len(X_val_raw)

X_train = X_all.iloc[:n_train]
X_val   = X_all.iloc[n_train:n_train+n_val]
X_test  = X_all.iloc[n_train+n_val:]

feature_columns = list(X_train.columns)

print(f"Features after encoding: {len(feature_columns)}")
print(f"Training:   {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")

## Step 6: Train Random Forest Model

In [ ]:
print("Training Random Forest Classifier...")
print("(This may take 1-2 minutes)")

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model trained successfully!")

## Step 7: Validation Set Results

In [ ]:
y_val_pred = model.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)

print(f"Validation Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")
print("\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred))

## Step 8: Test Set Results (Held-Out)

In [ ]:
y_test_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)
y_proba = model.predict_proba(X_test)
avg_confidence = float(np.mean(np.max(y_proba, axis=1)))

print(f"Test Accuracy (held-out): {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Average Confidence:       {avg_confidence:.4f} ({avg_confidence*100:.2f}%)")
print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred))

## Step 9: Confusion Matrix

In [ ]:
classes = model.classes_
cm = confusion_matrix(y_test, y_test_pred, labels=classes)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('Confusion Matrix — Raw Counts', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('Confusion Matrix — Normalized', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.suptitle(f'AssureX Model — Test Set Results (Accuracy: {test_acc*100:.1f}%)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrix.png")

## Step 10: Feature Importance

In [ ]:
importances = model.feature_importances_
fi_df = pd.DataFrame({'feature': feature_columns, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df, x='importance', y='feature', palette='viridis')
plt.title('Top 15 Most Important Features — AssureX Model',
          fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_importance.png")

## Step 11: Save Model

In [ ]:
model_package = {
    'model': model,
    'feature_columns': feature_columns,
    'classes': list(model.classes_),
    'model_name': 'AssureX-RandomForest-v2',
    'test_accuracy': test_acc,
    'val_accuracy': val_acc,
    'average_confidence': avg_confidence,
    'data_split': {
        'train': len(X_train),
        'val': len(X_val),
        'test': len(X_test)
    }
}

joblib.dump(model_package, 'assurex_claim_decision_model.pkl')
print("Model saved: assurex_claim_decision_model.pkl")
print(f"\nFinal Results:")
print(f"  Validation Accuracy: {val_acc*100:.2f}%")
print(f"  Test Accuracy:       {test_acc*100:.2f}%")
print(f"  Avg Confidence:      {avg_confidence*100:.2f}%")
print(f"  Classes:             {model_package['classes']}")

## Step 12: Download Files

In [ ]:
from google.colab import files

files.download('assurex_claim_decision_model.pkl')
files.download('confusion_matrix.png')
files.download('feature_importance.png')
files.download('class_distribution.png')

print("All files downloaded!")
print("\nNext steps:")
print("  1. Put assurex_claim_decision_model.pkl in:")
print("     ai_models/python_model/saved_models/")
print("  2. Submit confusion_matrix.png as training evidence")

---

## Summary

| Metric | Value |
|--------|-------|
| Algorithm | Random Forest (200 trees) |
| Dataset Size | 1,500 claims |
| Classes | Valid / Invalid / Manual Review |
| Train Split | 70% (1,049 rows) |
| Validation Split | 15% (226 rows) |
| Test Split | 15% (225 rows) |

**Evidence files generated:**
- `confusion_matrix.png` — Model performance visualization
- `feature_importance.png` — Top 15 features
- `class_distribution.png` — Dataset balance across splits
- `assurex_claim_decision_model.pkl` — Trained model file